# Paper combined - train the edit model, then run BCO experiments

**PART 1** trains the edit agent FROM SCRATCH on the 700-graph graded realistic curriculum (dup -> stub -> detour -> drop_cover -> subtle -> eval-mix -> clean) with the unified objective: 0.5*RTT + 0.5*WMC and a network-level adjustment budget (cap at target=0.2, W=10, paper mode, demand off; the BCO search uses the two-sided |adj-target| form). **PART 2** runs the paper experiments (E1-E6) using that fine-tuned model as `OUR_MODEL_PATH`. Algorithm budgets are set for a full top-to-bottom run that should fit into roughly 25-30 hours on the current CPU calibration; rerun the timing estimator cell after any budget change.

Run top-to-bottom: training must finish before the experiment config picks up the trained weights.


In [1]:
import torch

torch.__version__

'2.11.0+cpu'

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"   # select GPU 1
import sys, pickle, shutil, json, random as _random
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from hydra import compose, initialize_config_dir
from tqdm.auto import tqdm
from IPython.display import Image, display

from eval_lib.context import (ROOT_DIR, CFG_DIR, DATASETS_DIR,
                              MODEL_OUTPUTS_DIR, EDIT_MODEL_WEIGHTS_DIR)
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
from connectpt.routes_generator import utils as lrnu
from connectpt.routes_generator.improvement_learning import (
    _get_planned_current_routes, _make_route_context_state,
    load_raw_graphs_and_lc_routes, make_improvement_batch,
    rollout_lc_improvement, train_lc_improvement_cfg)
from connectpt.routes_generator.torch_utils import (
    get_batch_tensor_from_routes, dump_routes)
from connectpt.routes_generator.transit_time_estimator import (
    ROUTE_ACTION_EXTEND, ROUTE_ACTION_HALT, ROUTE_ACTION_TRIM_END,
    ROUTE_ACTION_TRIM_START, RouteGenBatchState)
from connectpt.routes_generator.citygraph_dataset import (
    STOP_KEY, DynamicCityGraphDataset)
from connectpt.routes_generator.bee_colony import get_adjustment_degrees
from torch_geometric.data import Batch
from eval_lib.results_io import save_table
from eval_lib import build_lc_cfg, run_lc, as_route_tensor
from eval_lib import plots as route_plots

pd.set_option("display.max_columns", None)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


In [ ]:
import os, sys
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display

from eval_lib.context import ROOT_DIR, EDIT_MODEL_WEIGHTS_DIR
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
from eval_lib import *                      # runners, builders, BENCHMARK_SPECS, ...
from eval_lib import _run_baseline          # private: not pulled by `import *`
from eval_lib import plots as route_plots
from connectpt.routes_generator.bee_colony import get_adjustment_degrees

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pd.set_option("display.max_columns", None)
print("device:", device)

# --- wall-clock instrumentation (per-epoch / per-iteration) ---
import time as _time
TIMING = {"epoch_s": None, "dataset_graph_s": None, "bco_iter_s": {}, "base_iter_s": {}}

# --- experiment-suite profile: ONE source of truth = cfg/experiments/suite*.yaml ---
# 'suite_smoke' = every experiment ON, single city, 1-iteration specs, TEMP_ outputs
# (a dry-run that never overwrites real files); 'suite' = the full paper run.
from eval_lib.suite import load_suite_config
from eval_lib.paper import set_paper_prefix
SUITE = load_suite_config("suite_smoke")
set_paper_prefix(SUITE.output_prefix)
print(f"suite profile={SUITE.profile} smoke={SUITE.smoke} prefix={SUITE.output_prefix!r} "
      f"cities={list(SUITE.cities)}")


## Configuration

`copy_full`, `copy_boundary`, `copy_mixed`, and `lc_clean` are equal-size
tiers.  The three corrupted tiers are generated from LC routes with the four
copy/subcopy mutations below.  Multiplicity grows across the curriculum up to
five routes on one stop-to-stop leg.

In [ ]:
# --- training configuration: cfg/train/edit_scratch[_smoke].yaml, selected by the
# suite profile loaded above (smoke -> tiny dataset, 1 iteration, TEMP_ outputs). ---
from experiments.training_lc import load_train_config
train_cfg = load_train_config("edit_scratch_smoke" if SUITE.smoke else "edit_scratch")
print(f"train cfg: run={train_cfg.run.name}, n_iterations={train_cfg.train_loop.n_iterations}, "
      f"dataset={train_cfg.data.dataset_dirname}, tiers={list(train_cfg.curriculum.tiers)}")

# Unified objective for the LIVE experiment-setup cells (benchmark loader, EKB).
# Read from the single source (cfg/objective/rtt_wmc_no_demand.yaml) via the
# library factory -- these are config values, not notebook-defined literals.
from connectpt.routes_generator.objectives import load_unified_objective
_OBJ = load_unified_objective()
CONNECTIVITY_MODE = _OBJ.connectivity_mode
DISABLED_COST_COMPONENTS = list(_OBJ.disabled_components)
UNIFIED_COST_WEIGHTS = _OBJ.weights
ADJ_WEIGHT, ADJ_TARGET = _OBJ.adj_weight, _OBJ.adj_target
ADJ_GAP, ADJ_MODE = _OBJ.adj_gap, _OBJ.adj_mode

## Generate the four-tier route-copy dataset

Every corruption copies a whole donor route or a contiguous donor subroute
into another route slot.  Prefix, suffix, and interior replacements preserve
the recipient length.  Candidates with repeated stops are rejected, so the
dataset teaches inter-route redundancy rather than synthetic self-loops.

In [ ]:
if SUITE.run.training:
    # Dataset generation: build the copy-tier dataset straight from the cfg
    # (experiments.training_lc.copytier_config reads cfg.dataset_gen / data /
    # curriculum; no N_GRAPHS / RAW_* / LC_COMBOS constants in the notebook).
    from experiments.training_lc import copytier_config, build_copytier_dataset

    seconds = build_copytier_dataset(copytier_config(train_cfg))
    if seconds is not None:
        TIMING["dataset_graph_s"] = seconds
        print(f"[timing] dataset gen: {seconds:.2f} s/graph "
              f"(N_GRAPHS={train_cfg.dataset_gen.n_graphs})")


## Load, split, and define the cumulative curriculum

In [ ]:
if SUITE.run.training:
    # Load graphs + the tier-stratified split for the dormant report cells.
    # Geometry/paths come from copytier_config(train_cfg); the split reproduces
    # TrainingDataModule.stratified_split (same logic EditTrainingRun uses).
    from connectpt.routes_generator.training import TrainingDataModule
    from experiments.training_lc import copytier_config

    dataset_cfg = copytier_config(train_cfg)
    data_module = TrainingDataModule(
        raw_graphs_path=dataset_cfg.subset_pkl, lc_results_dir=dataset_cfg.new_dataset_dir, device=device,
        min_route_len=dataset_cfg.min_route_len, max_route_len=dataset_cfg.max_route_len,
        target_n_routes=dataset_cfg.target_n_routes).setup()
    graphs, seed_routes = data_module.graphs, data_module.seed_routes
    meta_df = pd.read_csv(dataset_cfg.meta_csv)
    print(f"loaded {len(graphs)} graphs; seed_routes={tuple(seed_routes.shape)}")
    display(meta_df.groupby("tier")[[
        "applied_events", "redun_before", "redun_after",
        "max_leg_use_after", "d_un_after_pct"]].mean().round(3).reindex(dataset_cfg.tiers))

    tier_of = dict(zip(meta_df["graph_index"], meta_df["tier"]))
    (TRAIN_INDICES, VAL_INDICES, MONITOR_VAL_INDICES,
     train_by_tier, val_by_tier) = data_module.stratified_split(
        tier_of, dataset_cfg.tiers, train_fraction=float(train_cfg.data.train_fraction),
        n_val_per_tier=int(train_cfg.curriculum.n_val_per_tier),
        seed=int(train_cfg.data.split_seed))
    print(f"validation graphs={len(VAL_INDICES)}; monitor={len(MONITOR_VAL_INDICES)}")

    # curriculum stage spans (start,end,label) for the history-figure shading
    n_iter = int(train_cfg.train_loop.n_iterations)
    schedule = [(round(float(f) * n_iter), list(t), lab)
                 for f, t, lab in train_cfg.curriculum.schedule]

    def stage_spans():
        spans, prev = [], 0
        for until, _t, label in schedule:
            spans.append((prev + 1, until, label)); prev = until
        return spans


## Model and objective builders

Two helpers used by the fine-tune run below. `build_edit_run` composes the
hydra config, builds a **fresh** trim model + cost module, selects which of
the three cost components (`demand` / `route` / `connectivity`) are active,
and enables the optional adjustment-degree shaping through `adj_weight`.
`train_edit_run` wraps `train_lc_improvement_cfg` and returns the in-memory
history. Adjustment conditioning stays off, so the fine-tuned actor keeps the
same architecture as the RTT+connectivity checkpoint.


In [7]:
# Config-first model/cost builder for the dormant standalone balanced-eval cell.
# There is no bespoke "build_edit_run" anymore -- the real training runs via
# EditTrainingRun (cell 15). This re-exports the library helpers that build the
# edit model + unified cost config-first from train/edit + the objective YAML;
# the old 60-line builder (with vestigial adj_*/critic params it never used) and
# the rollout-kwargs helper now live in experiments/training_lc.py.
from experiments.training_lc import build_edit_model_and_cost, rollout_adjustment_kwargs

print("builders ready: build_edit_model_and_cost(), rollout_adjustment_kwargs() "
      "[config-first via experiments.training_lc]")


builders ready: build_edit_model_and_cost(), rollout_adjustment_kwargs() [config-first via experiments.training_lc]


## Clean LC baseline costs for history plots

Generate clean LC routes for the validation monitor graphs, score them with the same route/connectivity scalar used by the training-history plots, and save the result as a standalone CSV baseline.


In [ ]:
# Clean-LC baseline for the training-history figure. Knobs live in
# cfg.report.baseline; heavy logic in experiments.training_lc.clean_lc_baseline.
if SUITE.run.clean_lc_baseline:
    from experiments.training_lc import clean_lc_baseline, copytier_config
    dataset_cfg = copytier_config(train_cfg)
    if "graphs" not in globals() or "seed_routes" not in globals():
        graphs, seed_routes = load_raw_graphs_and_lc_routes(dataset_cfg.subset_pkl, dataset_cfg.new_dataset_dir)
    if "meta_df" not in globals():
        meta_df = pd.read_csv(dataset_cfg.meta_csv)
    clean_lc_baseline_df = clean_lc_baseline(
        train_cfg, graphs=graphs, seed_routes=seed_routes, meta_df=meta_df, device=device,
        baseline_path=MODEL_OUTPUTS_DIR / f"{train_cfg.run.name}_clean_lc_baseline_cost.csv")
    display(clean_lc_baseline_df.round(4))


## From-scratch training - route + connectivity + adj (W=10)

This run starts from the final-experiments RTT + connectivity checkpoint,
keeps `route` and `connectivity` active at 0.5/0.5 (demand off), and adds
network-level adjustment-budget reward shaping (cap at target=0.2, W=10, paper mode). The budget is 100
epochs over the current 500-graph copy-redundancy curriculum, batch 4, with
balanced per-tier validation monitoring. The resulting checkpoint is saved
under a new `adjcap` run name and is used by the evaluation cells below.

If `RESUME_FROM_CHECKPOINT=True`, this cell reloads the adj-finetune output
first; otherwise it initializes from `BASE_MODEL_PATH`.


In [ ]:
if SUITE.run.training:
    # Config-first training: EditTrainingRun builds model + cost + curriculum +
    # data from train_cfg (cfg/train/edit_scratch.yaml) and trains it.
    from pathlib import Path
    from connectpt.routes_generator.training import EditTrainingRun

    artifact = EditTrainingRun(train_cfg).run()
    history_df = artifact.history
    BEST_MODEL_PATH = Path(artifact.checkpoint_path)
    print(f"training done -> {BEST_MODEL_PATH} ({len(history_df)} epochs)")


## Train the edit model (cumulative curriculum)

Fine-tune the edit model over the cumulative curriculum (duplicate -> boundary
-> mixed -> covered -> clean). Training history is checkpointed to a partial CSV;
on resume the prior history is prepended.

In [ ]:
if SUITE.run.training:
    # History stitching + actor curves + TensorBoard mirroring (cfg-driven:
    # run name + scalar allow-list come from train_cfg.report).
    from experiments.training_lc import stitch_history, plot_training_history

    h, spans = stitch_history(
        prior_history_files=(),
        history_df=globals().get("history_df"),
        full_history_checkpoint=MODEL_OUTPUTS_DIR / f"{train_cfg.run.name}_training_history_partial.csv")
    if not spans and "stage_spans" in globals():
        spans = stage_spans()
    shade = plot_training_history(h, spans, train_cfg, model_outputs_dir=MODEL_OUTPUTS_DIR)
    print(f'[tensorboard] launch:  tensorboard --logdir "{MODEL_OUTPUTS_DIR / "tensorboard"}"')


## Critic diagnostics (per-component critic MSE / explained variance)

In [ ]:
if SUITE.run.training:
    from experiments.training_lc import plot_critic_metrics
    crit_table = plot_critic_metrics(h, shade)
    if crit_table is not None:
        display(crit_table)


## Evaluate balanced policy by tier

The table reports before/after route-level redundancy, `ATT`, `RTT`,
connectivity, and demand percentages by transfer bucket (`d0`, `d1`, `d2`,
`d_un`). `Adj(current, seed)` remains available only in the example plots.

In [ ]:
if SUITE.run.training:
    # Standalone balanced eval: rebuild model+cost from the checkpoint if the
    # training cell wasn't run, then evaluate per tier (cfg-driven).
    from experiments.training_lc import balanced_eval_by_tier

    if "model" not in globals() or "cost_obj" not in globals():
        _, cost_obj, model, _, BEST_MODEL_PATH = build_edit_model_and_cost(
            run_name=train_cfg.run.name, device=device,
            vary_weights=bool(train_cfg.cost.variable_weights),
            adj_weight=float(train_cfg.adjustment_degree_weight))
        if not BEST_MODEL_PATH.exists():
            raise FileNotFoundError(
                f"No trained checkpoint at {BEST_MODEL_PATH}. Train first, or check run.name.")
        model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
        model.eval()
        print(f"[eval] loaded trained model <- {BEST_MODEL_PATH.name} (training cell not run)")

    eval_df, visual_examples = balanced_eval_by_tier(
        train_cfg, model=model, cost_obj=cost_obj, device=device,
        graphs=graphs, seed_routes=seed_routes, val_by_tier=val_by_tier)
    display(eval_df)
    save_table(eval_df, f"{train_cfg.run.name}_eval_by_tier")
    vx_path = MODEL_OUTPUTS_DIR / f"{train_cfg.run.name}_visual_examples.pt"
    torch.save(visual_examples, vx_path)
    print(f"[eval] saved visual_examples -> {vx_path}")
    print("ATT/RTT/CONN are minutes; d0/d1/d2/d_un are demand percentages by transfer bucket.")


## Visual validation examples

For the balanced preference vector, show one seed and the corresponding edited
network from every tier.  The right-hand panels emphasize removed and added
segments relative to the corrupted seed.

In [ ]:
if SUITE.run.training:
    from experiments.training_lc import plot_balanced_examples
    if "visual_examples" not in globals() or not visual_examples:
        vx_path = MODEL_OUTPUTS_DIR / f"{train_cfg.run.name}_visual_examples.pt"
        if vx_path.exists():
            try:
                visual_examples = torch.load(vx_path, map_location="cpu", weights_only=False)
            except TypeError:
                visual_examples = torch.load(vx_path, map_location="cpu")
            print(f"[viz] loaded visual_examples <- {vx_path.name}")
        else:
            visual_examples = {}
    plot_balanced_examples(visual_examples, graphs, list(train_cfg.curriculum.tiers))


## Post-training convergence: RPC + type2 vs RPC + trim/extend (Mumford1, 100 iters)

Right after PART 1 training, run two of the 5-model BCO variants -- random
path-combiner (RPC) rebuild paired with the hand-designed `type2` edit vs the
freshly trained `trim/extend` edit bee -- on Mumford1 for 100 BCO iterations
(balanced alpha=0.5, adjustment penalty off) and plot the convergence history.

In [ ]:
if SUITE.run.training:
    # Post-PART-1 convergence: two 5-model BCO variants driven by the just-trained
    # edit checkpoint (city/iters/alpha from train_cfg.report.post_train). BCO
    # logic in training_lc; the notebook keeps only the convergence plot.
    import matplotlib.pyplot as plt
    from experiments.training_lc import post_training_convergence

    conv_df, curves = post_training_convergence(
        train_cfg, best_model_path=BEST_MODEL_PATH,
        benchmark_specs=BENCHMARK_SPECS, load_benchmark_graph=load_benchmark_graph)
    display(conv_df)

    if curves:
        post = train_cfg.report.post_train
        fig, ax = plt.subplots(figsize=(8, 5))
        for label, curve in curves.items():
            ax.plot(range(1, len(curve) + 1), curve, marker="o", ms=2, label=label)
        ax.set_xlabel("BCO iteration")
        ax.set_ylabel("best objective cost (lower = better)")
        ax.set_title(f"Post-training convergence -- {post.city} "
                     f"(alpha={post.alpha}, adj off, {post.iters} iters)")
        ax.grid(alpha=0.3); ax.legend()
        plt.show(); plt.close(fig)
        curves_df = pd.DataFrame({k: pd.Series(v) for k, v in curves.items()})
        curves_df.index.name = "bco_iteration"
        save_table(curves_df.reset_index(), f"posttrain_convergence_{post.city}")
    else:
        print("[post-train] no convergence history captured")


---
# PART 2 - BCO experiments (E1-E6)

Uses the model trained above (`BEST_MODEL_PATH`). All baselines are re-run live on a single seed with small budgets.

## Configuration

`SMOKE=True` -> Mandl only, 1 seed, tiny iteration budgets (just checks the
pipeline builds tables/figures). `SMOKE=False` runs Mandl and Mumford0-3
with the 25-30 hour budgeted settings below.


In [ ]:
# --- PART 2 experiment runtime setup (driven by the suite profile, SUITE) ---
# All experiment SELECTION + parameters live in cfg/experiments/suite*.yaml (read
# as SUITE in the setup cell). Here we only build the runtime objects the
# experiment cells need: the edit-model side-effects for the BCO bees, the
# per-city algo budgets, and the output prefix (already set from SUITE.output_prefix).
OUR_MODEL_PATH = EDIT_MODEL_WEIGHTS_DIR / SUITE.model.edit_checkpoint
import eval_lib.helpers as _eh
_eh.EDIT_MODEL_WEIGHTS_PATH = OUR_MODEL_PATH
_eh.EDIT_MODEL_N_ADJ_COND_FEATS = int(SUITE.model.edit_adj_cond_feats)
import eval_lib.baselines as _eb
_eb.BENCHMARK_INIT_MODE = SUITE.benchmark_init_mode
if SUITE.quick:
    _eh.BCO_N_ITERATIONS = 5

# Per-city algorithm budgets (SA/HH/GA/BCO iteration counts, populations, the
# Mandl BCO temperature schedule) live in cfg/search/budgets.yaml.
from eval_lib.budgets import make_algo_settings, load_budgets
_BUDGETS = load_budgets()
HH_MAX_REPAIR_ITERS = _BUDGETS["hh_max_repair_iters_default"]
E2_BCO_ITERATIONS = _BUDGETS["e2_bco_iterations"]
algo_settings = make_algo_settings(SUITE.smoke, SUITE.quick, budgets=_BUDGETS)

print("cities:", list(SUITE.cities), "| our model:", OUR_MODEL_PATH.name,
      "| exists:", OUR_MODEL_PATH.exists())
print("output prefix:", repr(SUITE.output_prefix), "| enabled experiments:", dict(SUITE.run))
print("E1 narrow:", list(SUITE.e1.cities), "| alphas", list(SUITE.e1.alpha_grid),
      "| target", SUITE.e1.adj_target, "| E2 BCO iters:", E2_BCO_ITERATIONS)


## Shared metrics: adjustment-degree vs seed + redundancy

In [ ]:
from omegaconf import OmegaConf
from tqdm.auto import tqdm

# Generic plumbing lives in eval_lib (paper.py / route_copies.py). Experiment
# configs are config-first (cfg/experiments/*.yaml via run_experiment); this cell
# keeps only the shared LC-init pipeline.
from eval_lib.paper import (UNIFIED_ADJ, PAPER_DIR, bco_cfg_set,
                            set_cfg_value as _set_cfg_value,
                            unify_weights as _unify_weights,
                            eval_routes_cfg as _eval_routes_cfg,
                            ravel_hist as _ravel_hist,
                            paper_row as _row, full_metrics as _full_metrics,
                            adj_vs_init, redundancy_pct, conn_metric,
                            save_paper_table, reset_paper_table,
                            append_paper_row, save_paper_fig, save_paper_routes)
from eval_lib.route_copies import (inject_realistic_tier, pad_routes,
                                   uncovered_demand_pct)

# LC init optimizes the same unified objective (RTT + WMC, demand off).
LC_INIT_WEIGHTS = UNIFIED_COST_WEIGHTS



# --- experiment init: LC construction base + the REALISTIC corruption tier
#     (eval_lib.route_copies.REALISTIC_TIER_CFG): a couple of street-valid
#     partial duplicates, 1-2 detour routes the agent should straighten, and a
#     few dropped low-demand stops -- instead of the old covered_dup tier that
#     cloned half the network and glued phantom (non-street) legs. ---
import random as _random
LC_INIT_SEED = 0
EXP_INIT_TIER = "realistic"   # tier injected into the experiment init network
_LC_INIT_CACHE = {}


def _lc_base_routes(spec):
    """Clean LC construction routes for a city (no tier corruption)."""
    tensors = load_benchmark_tensors(spec["city"])
    cfg = build_lc_cfg(run_name=f"lc_init_{spec['city']}", n_routes=spec["n_routes"],
                       min_route_len=spec["min_route_len"],
                       max_route_len=spec["max_route_len"],
                       connectivity_mode=CONNECTIVITY_MODE, **LC_INIT_WEIGHTS)
    _set_cfg_value(cfg, "experiment.seed", int(LC_INIT_SEED))
    routes = run_lc(cfg, tensors=tensors, run_name_prefix=f"lc_init_{spec['city']}_", n_samples=1)[3]
    return tensors, pad_routes(routes, spec["n_routes"], spec["max_route_len"])


def _e1_init_from_file(spec):
    """Variant A: reuse a saved init network for a city -- load the
    'Initial ...' routes from
    final_main_unified_<city><E1_INIT_SOURCE_STEM>_routes.pt so init is
    machine-independent and identical across methods. Returns a padded
    (1, n_routes, max_len) tensor, or None to fall back to LC generation."""
    stem = SUITE.e1.init_source_stem
    if not stem:
        return None
    path = PAPER_DIR / f"final_main_unified_{spec['city']}{stem}_routes.pt"
    if not path.exists():
        return None
    try:
        dump = torch.load(path, map_location="cpu", weights_only=False)
        routes = dump.get("routes", {})
        key = next((k for k in routes if str(k).startswith("Initial")), None)
        if key is None:
            return None
        return pad_routes(routes[key], spec["n_routes"], spec["max_route_len"])[None]
    except Exception as exc:
        print(f"[init] {spec['city']}: file load failed ({exc}); regenerating via LC")
        return None

def load_benchmark_graph(spec, init_mode=None):     # overrides eval_lib.load_benchmark_graph
    city = spec["city"]
    if city not in _LC_INIT_CACHE:
        _file_init = _e1_init_from_file(spec)
        if _file_init is not None:
            tensors = load_benchmark_tensors(city)
            init = as_route_tensor(_file_init)
            if init.ndim == 2:
                init = init[None]
            _LC_INIT_CACHE[city] = (tensors, init)
            _dun = uncovered_demand_pct(init[0], tensors["demand"],
                                        tensors["node_locs"].shape[0])
            print(f"[init] {city}: loaded from saved "
                  f"final_main_unified_{city}{SUITE.e1.init_source_stem} dump "
                  f"-> redun={redundancy_pct(init):.1f}% d_un={_dun:.1f}%")
        else:
            tensors, clean = _lc_base_routes(spec)
            rng = _random.Random(LC_INIT_SEED)
            init, _applied = inject_realistic_tier(
                clean, rng, spec["min_route_len"], spec["max_route_len"],
                street_adj=tensors["street_adj"], demand=tensors["demand"],
                n_nodes=tensors["node_locs"].shape[0])
            init = as_route_tensor(init)
            if init.ndim == 2:
                init = init[None]                      # -> (1, n_routes, max_len) for runners
            _LC_INIT_CACHE[city] = (tensors, init)
            _dun = uncovered_demand_pct(init[0], tensors["demand"],
                                        tensors["node_locs"].shape[0])
            print(f"[init] {city}: LC base + tier '{EXP_INIT_TIER}' "
                  f"({dict(_applied)}) -> redun={redundancy_pct(init):.1f}% "
                  f"d_un={_dun:.1f}%")
    return _LC_INIT_CACHE[city]


# Unified objective (RTT + WMC + two-sided |adj-target|) shared by E1u and MACSA.




## EKB case study

Load the Ekaterinburg instance, inspect its projected coordinates, and render the supplied seed routes as a regular matplotlib figure.


In [ ]:
import eval_lib.paper as _paper  # output prefix (TEMP_ on smoke)
# EKB case study: load the active instance and render its seed route network.
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from eval_lib import plots as route_plots
from eval_lib.ekb import (EKB_COORD_CRS, EKB_STATIC_MAP_PATH, ekb_connectivity_stats,
                          ekb_route_stats, make_ekb_street_underlay_adj, project_ekb_coords)
from experiments import ekb as _ekb

if SUITE.run.ekb_case_study:
    EKB_CASE = _ekb.case_from_cfg(SUITE.ekb)
    _stats = ekb_route_stats(EKB_CASE.init)
    _latlon = project_ekb_coords(EKB_CASE.tensors["node_locs"], EKB_COORD_CRS)
    print("EKB coords:", tuple(EKB_CASE.tensors["node_locs"].shape), "CRS", EKB_COORD_CRS,
          "lat", (round(float(_latlon[:, 0].min()), 4), round(float(_latlon[:, 0].max()), 4)),
          "lon", (round(float(_latlon[:, 1].min()), 4), round(float(_latlon[:, 1].max()), 4)))
    _conn = ekb_connectivity_stats(EKB_CASE.tensors, EKB_CASE.init)
    print("EKB spec:", EKB_CASE.spec, "| route stats:", _stats)
    print("EKB connectivity:", {"components": _conn["n_components"],
          "isolated_nodes": _conn["isolated_nodes"], "symmetric": _conn["symmetric"],
          "cross_component_demand_pct": round(_conn["cross_component_demand_pct"], 4)})

    _subtitle = (f"{_stats['n_routes']} routes | {_stats['unique_stops']} stops | "
                 f"len {_stats['min_len']}-{_stats['max_len']} mean={_stats['mean_len']:.1f}")
    if SUITE.ekb.score_seed:
        _res = _ekb.score_routes(EKB_CASE, EKB_CASE.init, "seed_eval",
                                 adj_target=_ekb.nbco_adj_target(SUITE.ekb),
                                 force_cpu=bool(SUITE.ekb.force_cpu))
        _seed_row = _row("EKB", "Initial EKB routes", "ekb_seed", _res[1], _res[3], EKB_CASE.init)
        display(pd.DataFrame([_seed_row]).round(4))
        _subtitle += (f"\ncost={_seed_row['cost']:.3f}  RTT={_seed_row['RTT']:.0f}  "
                      f"WMC={_seed_row['WMC']:.2f}  redun={_seed_row['redun%']:.0f}%")

    _rr = EKB_CASE.init[0] if EKB_CASE.init.ndim == 3 else EKB_CASE.init
    _adj = make_ekb_street_underlay_adj(EKB_CASE.tensors["street_adj"])
    fig, ax = plt.subplots(1, 1, figsize=(10.5, 10), constrained_layout=True)
    route_plots.plot_plain_route_set(
        ax, _rr, EKB_CASE.tensors["node_locs"], _adj,
        title=f"EKB {EKB_CASE.case_tag} supplied routes", subtitle=_subtitle,
        palette="tab20", with_overlap_curves=True, show_node_labels=False,
        node_size=int(SUITE.ekb.route_fig_node_size))
    fig.suptitle(f"EKB {EKB_CASE.case_tag} seed route network", fontsize=15, fontweight="bold")
    _map = (EKB_STATIC_MAP_PATH if EKB_CASE.case_tag == "full"
            else EKB_STATIC_MAP_PATH.with_name(f"ekb_{EKB_CASE.case_tag}_seed_routes_static.png"))
    _map = _map.with_name(_paper.PAPER_PREFIX + _map.name)
    _map.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(_map, dpi=220, bbox_inches="tight")
    print(f"[EKB] static route map saved -> {_map}")
    plt.show(); plt.close(fig)


### EKB NBCO GNN + trim/extend

Run our NBCO variant on the Ekaterinburg instance and draw before/after route sets in both plain and diff modes. Overlapping route edges are rendered as curved arcs so duplicate coverage stays visible instead of collapsing into one line.


In [ ]:
# EKB NBCO: GNN rebuild + trim/extend edit bees, then before/after + diff figures.
# The run (scoring, BCO, caching, live best-checkpoint, tensorboard) lives in
# experiments.ekb.run_ekb_nbco; config comes from SUITE.ekb.
import matplotlib.pyplot as plt
from IPython.display import display
from eval_lib import plots as route_plots
from eval_lib.ekb import make_ekb_street_underlay_adj
from experiments import ekb as _ekb

if SUITE.run.ekb_nbco:
    EKB_CASE = _ekb.case_from_cfg(SUITE.ekb)
    print(f"[EKB NBCO] active case={EKB_CASE.case_tag} spec={EKB_CASE.spec}")
    EKB_NBCO = _ekb.run_ekb_nbco(
        EKB_CASE, ekb_cfg=SUITE.ekb,
        n_bees=int(algo_settings("EKB")["bco_bees"]), seed=SUITE.seeds[0],
        model_outputs_dir=MODEL_OUTPUTS_DIR)
    display(EKB_NBCO.df.round(4))

    _final_key = next((k for k in EKB_NBCO.results if "Our NBCO" in k), None)
    if _final_key is not None:
        _init_rr = EKB_CASE.init[0] if EKB_CASE.init.ndim == 3 else EKB_CASE.init
        _final_rt = as_route_tensor(EKB_NBCO.results[_final_key])
        _final_rr = _final_rt[0] if _final_rt.ndim == 3 else _final_rt
        _adj = make_ekb_street_underlay_adj(EKB_CASE.tensors["street_adj"])
        _ov, _ns = bool(SUITE.ekb.nbco.overlap_curves), int(SUITE.ekb.nbco.node_size)
        _sub = lambda lab: _ekb.metric_subtitle(EKB_NBCO.df, lab)

        fig, axes = plt.subplots(1, 2, figsize=(18, 8.5), squeeze=False, constrained_layout=True)
        route_plots.plot_plain_route_set(
            axes[0, 0], _init_rr, EKB_CASE.tensors["node_locs"], _adj,
            title=f"EKB {EKB_CASE.case_tag} initial routes", subtitle=_sub("Initial EKB routes"),
            palette="tab20", with_overlap_curves=_ov, show_node_labels=False, node_size=_ns)
        route_plots.plot_plain_route_set(
            axes[0, 1], _final_rr, EKB_CASE.tensors["node_locs"], _adj,
            title=_final_key, subtitle=_sub(_final_key),
            palette="tab20", with_overlap_curves=_ov, show_node_labels=False, node_size=_ns)
        fig.suptitle(f"EKB {EKB_CASE.case_tag} NBCO GNN + trim/extend: plain before / after",
                     fontsize=15, fontweight="bold")
        plt.show(); plt.close(fig)

        fig, ax = plt.subplots(1, 1, figsize=(10.5, 10), constrained_layout=True)
        route_plots.plot_route_diff(
            ax, _final_rr, _init_rr, EKB_CASE.tensors["node_locs"], _adj,
            title=f"EKB {EKB_CASE.case_tag} diff: NBCO vs initial",
            palette="tab20", with_overlap_curves=_ov, show_node_labels=False, node_size=_ns)
        fig.suptitle(f"EKB {EKB_CASE.case_tag} NBCO GNN + trim/extend: diff vs initial",
                     fontsize=15, fontweight="bold")
        plt.show(); plt.close(fig)
else:
    print("[EKB NBCO] skipped (SUITE.run.ekb_nbco=False)")


# EKB alpha sweep

Run only the Ekaterinburg alpha sweep with adjustment target 0.3. Results are written to a separate table and route dump so existing paper results are not overwritten.


In [ ]:
# EKB alpha sweep: route/connectivity trade-off at a fixed adjustment target.
# The sweep loop lives in experiments.ekb.run_ekb_alpha_sweep; config = SUITE.ekb.
from IPython.display import display
from experiments import ekb as _ekb

if SUITE.run.ekb_alpha_sweep:
    EKB_SWEEP_CASE = _ekb.case_from_cfg(SUITE.ekb)
    EKB_SWEEP_DF, EKB_SWEEP_ROUTES = _ekb.run_ekb_alpha_sweep(
        EKB_SWEEP_CASE, ekb_cfg=SUITE.ekb,
        n_bees=int(algo_settings("EKB")["bco_bees"]), seed=SUITE.seeds[0])
    display(EKB_SWEEP_DF.round(4))
else:
    print("[EKB sweep] skipped (SUITE.run.ekb_alpha_sweep=False)")


### EKB best solution (overwritten each run)

The best achieved EKB network (BCO incumbent) is written to fixed-name `artifacts/paper_results/final_ekb_best_solution.{csv,_routes.pt}` and overwritten on every run.

In [ ]:
# Save the single BEST achieved EKB solution (routes + characteristics) to a
# FIXED stem, overwritten each run. run_ekb_nbco already live-checkpoints the
# incumbent; this persists the final best "Our NBCO" row from the result table.
EKB_BEST_STEM = "final_ekb_best_solution"
if not SUITE.run.ekb_nbco or "EKB_NBCO" not in globals() or EKB_NBCO.df.empty:
    print("[EKB best] skipped (EKB NBCO not run).")
else:
    _best_key = next((k for k in EKB_NBCO.results if "Our NBCO" in k), None)
    if _best_key is None:
        print("[EKB best] no Our NBCO solution to save.")
    else:
        _best_routes = as_route_tensor(EKB_NBCO.results[_best_key])
        _best_row = EKB_NBCO.df[EKB_NBCO.df["method"] == _best_key].round(6)
        save_paper_table(_best_row, EKB_BEST_STEM)
        save_paper_routes(
            EKB_BEST_STEM,
            {"Initial EKB routes": as_route_tensor(EKB_CASE.init), _best_key: _best_routes},
            EKB_CASE.tensors["node_locs"], EKB_CASE.tensors["street_adj"],
            meta={"city": "EKB", "best_method": _best_key, "case_tag": EKB_CASE.case_tag,
                  "source_table": EKB_NBCO.table_stem})
        display(_best_row)
        print(f"[EKB best] overwritten -> {EKB_BEST_STEM}.csv + {EKB_BEST_STEM}_routes.pt "
              f"(method={_best_key})")


In [ ]:
# Mumford0 neural-BCO variants (Our NBCO vs Trim 12 + extend 12, alpha=0.5) --
# unified via run_experiment. The inline build_bco_cfg variant dicts, the RL-only
# probe and the bespoke 2-panel plot (~290 lines) collapse to a config-first
# m0_variants.yaml spec + run_experiment + render_report.
M0_NBCO_TABLE = ("final_mumford0_nbco_variants_alpha05_target03_iter200"
                 if SUITE.run.m0_nbco_variants else
                 "final_mumford0_nbco_variants_alpha05_target03_iter200_skipped")

if SUITE.run.m0_nbco_variants:
    from eval_lib.experiment_runner import run_experiment, load_experiment_spec
    from eval_lib.experiment_report import render_report

    result = run_experiment(load_experiment_spec("m0_variants_smoke" if SUITE.smoke else "m0_variants"))
    report = render_report(result)
    display(report.table)
    save_paper_table(result.table.round(4), M0_NBCO_TABLE)
    for fig in report.figures.values():
        display(fig)
else:
    print("Mumford0 NBCO variants skipped (suite.run.m0_nbco_variants=False)")


## E2 - Pareto fronts on RTT x WMC (Mumford0, covered_dup tier)

Both panels run on **Mumford0** from the **covered_dup** init tier (removable
redundancy), optimizing `alpha*RTT + (1-alpha)*WMC + adj(|.-target|)` with
`use_weighted_connectivity=True`. `alpha` = `route_time_weight` (so alpha=1 ->
pure RTT, alpha=0 -> pure WMC).

- **Fig 1 (our model):** RTT x WMC Pareto front of `GNN + trim/extend`, swept
  over `alpha` x `adj_target` (one curve per target).
- **Fig 2 (4-model comparison):** `{GNN, RPC} x {trim/extend, type2}`, alpha-
  swept at a fixed `adj_target=ADJ_TARGET` -> shows the trim/extend
  bee's contribution to the RTT x WMC front.

In [22]:
# === E2 -- Pareto fronts on RTT x WMC (Mumford1, realistic init tier) ===
E2_CITY = "Mumford1" if not SUITE.smoke else "Mandl"
E2_ALPHA_GRID = [round(0.25 * i, 2) for i in range(5)] if not SUITE.smoke else [0.0, 0.5, 1.0]
_E2_TARGET_GRID_FULL = [round(0.2 * i, 2) for i in range(1, 6)] if not SUITE.smoke else [0.2, 0.6]
E2_TARGET_GRID = _E2_TARGET_GRID_FULL if SUITE.run.e2_our_pareto else []
E2_FIX_TARGET = min(_E2_TARGET_GRID_FULL, key=lambda _t: abs(_t - 0.2))  # alpha-slice target
E2_FIX_ALPHA = min(E2_ALPHA_GRID, key=lambda _a: abs(_a - 0.5))    # target-slice alpha
E2_FIXED_TARGET = E2_FIX_TARGET
E2_ADJ_OBJECTIVE = "target"
E2_DEFAULT_ADJ_WEIGHT = float(ADJ_WEIGHT)
E2_FIG2_ADJ_WEIGHT = 0.0
print("E2 grid:", {"city": E2_CITY, "alpha": E2_ALPHA_GRID,
                   "adj_target": E2_TARGET_GRID, "fixed_target": E2_FIXED_TARGET,
                   "iters": E2_BCO_ITERATIONS,
                   "adj_objective": E2_ADJ_OBJECTIVE,
                   "run_our_pareto": SUITE.run.e2_our_pareto,
                   "run_5model": SUITE.run.e2_5model})


E2 grid: {'city': 'Mandl', 'alpha': [0.0, 0.5, 1.0], 'adj_target': [0.2, 0.6], 'fixed_target': 0.2, 'iters': 200, 'adj_objective': 'target', 'run_our_pareto': True, 'run_5model': True}


In [ ]:
# E2 Our-NBCO Pareto sweep (alpha x adj_target) on Mumford1 -- unified via
# run_experiment (2D sweep). The ~210-line inline ctx/run/plot helpers + the
# our-pareto loop collapse to the config-first e2_our_pareto.yaml spec.
e2_suffix = ("_smoke" if SUITE.smoke else "") + SUITE.e2.table_suffix

if SUITE.run.e2_our_pareto:
    from eval_lib.experiment_runner import run_experiment, load_experiment_spec
    from eval_lib.experiment_report import render_report

    result = run_experiment(load_experiment_spec("e2_our_pareto_smoke" if SUITE.smoke else "e2_our_pareto"))
    report = render_report(result)
    display(report.table)
    save_paper_table(result.table.round(4), f"final_e2_our_pareto_{E2_CITY}{e2_suffix}")
    for fig in report.figures.values():
        display(fig)
else:
    print(f"[E2] {E2_CITY}: our-model Pareto sweep skipped (suite.run.e2_our_pareto=False)")


In [ ]:
# E2 Fig-2: 5-model RTT x WMC comparison (GNN/RPC x trim-extend/type2 +
# trim12+extend12) on Mumford1, alpha-swept with adjustment OFF -- unified via
# run_experiment. The inline model-spec loop + config builders + plotting
# (~70 lines) become the config-first e2_5model.yaml spec.
e2_5model_table = f"final_e2_5model_{E2_CITY}{e2_suffix}"
if SUITE.run.e2_5model:
    from eval_lib.experiment_runner import run_experiment, load_experiment_spec
    from eval_lib.experiment_report import render_report

    result = run_experiment(load_experiment_spec("e2_5model_smoke" if SUITE.smoke else "e2_5model"))
    report = render_report(result)
    display(report.table)
    save_paper_table(result.table.round(4), e2_5model_table)
    for fig in report.figures.values():
        display(fig)
    E2_ABL_DF = result.table
else:
    print(f"[E2] {E2_CITY}: 5-model alpha sweep skipped (suite.run.e2_5model=False)")
    E2_ABL_DF = pd.DataFrame()


### E2 Route Visualisation

Draw selected `alpha x adj_target` route sets from the E2 our-model sweep. The first grid shows the route sets directly; the second grid highlights changes against the LC initial network.

In [ ]:
# Visualise E2 our-model routes -- unified via the atomic viz.plot_routes_grid
# (plain grid + diff-vs-Initial grid). Reads the dump saved by the E2 pareto cell.
import torch
from eval_lib import as_route_tensor, viz
from eval_lib.paper import paper_path

if SUITE.run.e2_route_viz:
    e2_suffix = ("_smoke" if SUITE.smoke else "") + SUITE.e2.table_suffix
    dump_path = paper_path(f"final_e2_our_pareto_{E2_CITY}{e2_suffix}_routes.pt")
    if not dump_path.exists():
        print(f"[E2 route viz] skipped: no dump {dump_path.name} (run the E2 our-pareto cell first)")
    else:
        dump = torch.load(dump_path, weights_only=False)
        route_sets = {label: as_route_tensor(rt) for label, rt in dump["routes"].items()}
        ref_label = next((m for m in route_sets if "Initial" in m), list(route_sets)[0])
        display(viz.plot_routes_grid(route_sets, dump["coords"], dump["street_adj"],
                                     title=f"E2 {E2_CITY}: our-model routes"))
        display(viz.plot_routes_grid(route_sets, dump["coords"], dump["street_adj"],
                                     diff_against=ref_label, title=f"E2 {E2_CITY}: diff vs {ref_label}"))
else:
    print("E2 route viz skipped (suite.run.e2_route_viz=False)")


## E1 narrow -- neural BCO vs Our NBCO alpha sweep

Runs only two methods on every benchmark city (`Mandl`, `Mumford0`, `Mumford1`, `Mumford2`, `Mumford3`):
`neural BCO` and `Our NBCO (GNN rebuild + trim/extend)`. For each method we run
`alpha in {0, 0.5, 1}` with `adjustment_degree_target=0.5` and 100 BCO iterations.

All SA / GA / HH / heuristic BCO / NSGA-II / trim-only branches are disabled here. Tables are saved as
`paper_results/final_main_unified_<city><E1U_TABLE_SUFFIX>.csv` and a combined table as
`final_main_unified_comparison<E1U_TABLE_SUFFIX>.csv`.


In [ ]:
# E1 narrow: neural BCO vs Our NBCO alpha sweep -- unified via run_experiment.
# Each e1_<city>.yaml spec lists the data source, the two methods (captured
# bee-mix configs) and the alpha sweep; run_experiment iterates methods x alpha
# and render_report builds the table (was a ~190-line hand-rolled loop).
from eval_lib.experiment_runner import run_experiment, load_experiment_spec
from eval_lib.experiment_report import render_report

e1_suffix = ("_smoke" if SUITE.smoke else "") + SUITE.e1.table_suffix
all_rows = []
for city in SUITE.e1.cities:
    print(f"=== {city} E1 narrow: neural BCO + Our NBCO, alphas={list(SUITE.e1.alpha_grid)}, "
          f"target={SUITE.e1.adj_target}, iter={SUITE.e1.bco_iterations} ===", flush=True)
    result = run_experiment(load_experiment_spec(f"e1_{city.lower()}" + ("_smoke" if SUITE.smoke else "")))
    report = render_report(result)
    display(report.table)
    save_paper_table(result.table.round(3), f"final_main_unified_{city}{e1_suffix}")
    all_rows += result.table.to_dict("records")

unified_df = pd.DataFrame(all_rows).round(3)
if not unified_df.empty:
    display(unified_df)
    save_paper_table(unified_df, "final_main_unified_comparison" + e1_suffix)
else:
    print("E1 narrow skipped: no cities configured (suite.e1.cities is empty).")


### Route visualisation (unified run: RTT + WMC + adj for all methods)

Drawn from the **unified** dumps (`final_main_unified_<city>`). Layout: OD demand -> init -> methods (plain & diff vs init), plus a focused 1x4 (our routes / init / neural-BCO diff / our-NBCO diff).

In [ ]:
# Route-set visualisation from saved E1 route dumps -- unified via the atomic
# viz.plot_routes_grid (plain panels + a diff-vs-Initial grid, one style).
import torch
from eval_lib import as_route_tensor, viz
from eval_lib.paper import paper_path

VIZ_CITY = "Mumford0"
VIZ_NCOL = 3
suffix = ("_smoke" if SUITE.smoke else "") + SUITE.e1.table_suffix
dump_path = paper_path(f"final_main_unified_{VIZ_CITY}{suffix}_routes.pt")
if not dump_path.exists():
    print(f"[route viz] skipped: no dump {dump_path.name} (run E1u for {VIZ_CITY} first)")
else:
    dump = torch.load(dump_path, weights_only=False)
    route_sets = {label: as_route_tensor(rt) for label, rt in dump["routes"].items()}
    coords, street_adj = dump["coords"], dump["street_adj"]
    ref_label = next((m for m in route_sets if "Initial" in m), list(route_sets)[0])
    display(viz.plot_routes_grid(route_sets, coords, street_adj, ncols=VIZ_NCOL,
                                 title=f"{VIZ_CITY}: route sets"))
    display(viz.plot_routes_grid(route_sets, coords, street_adj, ncols=VIZ_NCOL,
                                 diff_against=ref_label, title=f"{VIZ_CITY}: diff vs {ref_label}"))


## MACSA Table B -- paper routes + Our NBCO alpha sweep

This section replaces the old generic MACSA probe with the exact Mandl-8 Table-B workflow used for the paper figures. It reads the fixed route sets from `datasets/MACSA_data/mandl_8/routes_*.txt`, scores them with the same E1-style metric plumbing, then runs Our NBCO from the original network with `adjustment_degree_target = adj(MACSA, original)`.

Outputs:
- `final_macsa_mandl8_alpha_sweep_iter100.*`: Our NBCO alpha sweep, `alpha=0.0..1.0` step `0.1`, 100 BCO iterations per run.
- `final_macsa_mandl8_tableb.*`: Table-B paper methods plus the best Our NBCO sweep solution that beats MACSA on both RTT and WMC. The old cap-mode row is intentionally not included.


In [ ]:
# MACSA Table-B case study: thick helpers now live in experiments/macsa.py
# (one-off paper experiment kept out of the reusable library). Configure the
# runtime values, then import the helpers + scenario constants by their bare
# names so the orchestration cells below read unchanged.
import sys
from pathlib import Path

_MACSA_NOTEBOOK_DIR = Path("examples/route_generator").resolve()
if str(_MACSA_NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(_MACSA_NOTEBOOK_DIR))

from experiments import macsa

_cfg = macsa.configure(
    smoke=SUITE.smoke,
    our_model_path=OUR_MODEL_PATH,
    seeds=SUITE.seeds,
)
from experiments.macsa import *  # noqa: F401,F403  (helpers + MACSA_* constants)

print("MACSA Table-B config:", _cfg)


In [29]:
# Score the fixed Mandl-8 Table-B routes from the MACSA paper.
if not SUITE.run.macsa:
    MACSA_TABLEB_DF = pd.DataFrame()
    MACSA_TABLEB_ROUTES = {}
    print("MACSA Table-B skipped (RUN_MACSA_EXPERIMENTS=False).")
else:
    MACSA_TENSORS = load_macsa_tensors(MACSA_SCENARIO_DIR)
    MACSA_COORDS = MACSA_TENSORS["node_locs"]
    MACSA_STREET_ADJ = MACSA_TENSORS["street_adj"]
    MACSA_DEMAND = MACSA_TENSORS["demand"]
    MACSA_RAW_ROUTES = {
        MACSA_METHOD_TITLE[key]: macsa_read_routes_0indexed(MACSA_SCENARIO_DIR / f"routes_{key}.txt")
        for key in MACSA_METHOD_ORDER
    }
    MACSA_SPEC = macsa_build_spec(MACSA_RAW_ROUTES, int(MACSA_COORDS.shape[0]))
    MACSA_TABLEB_ROUTES = {
        method: macsa_pad_routes(routes, MACSA_SPEC["n_routes"], MACSA_SPEC["max_route_len"])
        for method, routes in MACSA_RAW_ROUTES.items()
    }
    MACSA_SEED_ROUTES = MACSA_TABLEB_ROUTES[MACSA_REF_METHOD]
    MACSA_ADJ_TARGET_FROM_MACSA = float(adj_vs_init(MACSA_TABLEB_ROUTES["MACSA"], MACSA_SEED_ROUTES))

    MACSA_TABLEB_ROWS = []
    for method in MACSA_TABLEB_ROUTES:
        row, scored = macsa_score_routes(
            method, "macsa_table_b", MACSA_TABLEB_ROUTES[method],
            seed_routes=MACSA_SEED_ROUTES, tensors=MACSA_TENSORS, spec=MACSA_SPEC,
            alpha=MACSA_TABLEB_EVAL_ALPHA,
            adj_target=MACSA_TABLEB_EVAL_ADJ_TARGET,
            adj_objective=MACSA_TABLEB_EVAL_ADJ_OBJECTIVE)
        row.update(alpha=np.nan, run_alpha=np.nan, n_iterations=np.nan,
                   macsa_adj_target=MACSA_ADJ_TARGET_FROM_MACSA)
        MACSA_TABLEB_ROWS.append(row)
        MACSA_TABLEB_ROUTES[method] = scored
        print(f"  {method:20} ATT={row['ATT']:.2f} RTT={row['RTT']:.0f} "
              f"WMC={row['WMC']:.2f} adj={row['adj_vs_seed']:.3f} cost={row['cost']:.3f}")

    MACSA_TABLEB_DF = pd.DataFrame(MACSA_TABLEB_ROWS).round(6)
    save_paper_table(MACSA_TABLEB_DF, MACSA_ARTICLE_STEM)
    save_paper_routes(MACSA_ARTICLE_STEM, MACSA_TABLEB_ROUTES,
                      MACSA_COORDS, MACSA_STREET_ADJ,
                      meta={"scenario": MACSA_SCENARIO_NAME,
                            "ref_method": MACSA_REF_METHOD,
                            "eval_alpha": MACSA_TABLEB_EVAL_ALPHA,
                            "eval_adj_target": MACSA_TABLEB_EVAL_ADJ_TARGET,
                            "macsa_adj_target": MACSA_ADJ_TARGET_FROM_MACSA})
    print(f"MACSA adj target from Table-B MACSA vs original: {MACSA_ADJ_TARGET_FROM_MACSA:.6f}")
    display(MACSA_TABLEB_DF)


NameError: name 'load_macsa_tensors' is not defined

In [ ]:
# Run Our NBCO alpha sweep: alpha=0.0..1.0 step 0.1, exactly 100 BCO iterations each.
if not SUITE.run.macsa or MACSA_TABLEB_DF.empty:
    MACSA_SWEEP_DF = pd.DataFrame()
    MACSA_SWEEP_ROUTES = {}
    print("MACSA alpha sweep skipped: no scored Table-B routes.")
else:
    sweep_csv = PAPER_DIR / f"{MACSA_SWEEP_STEM}.csv"
    sweep_routes_path = PAPER_DIR / f"{MACSA_SWEEP_STEM}_routes.pt"
    cached_routes = {}
    cached_duration = {}
    if sweep_routes_path.exists() and sweep_csv.exists() and not MACSA_SWEEP_FORCE_RERUN:
        try:
            payload = torch.load(sweep_routes_path, map_location="cpu", weights_only=False)
            meta = payload.get("meta", {})
            if int(meta.get("bco_iterations", -1)) == int(MACSA_SWEEP_BCO_ITERATIONS):
                cached_routes = {k: as_route_tensor(v) for k, v in payload.get("routes", {}).items()}
                cache_df = pd.read_csv(sweep_csv)
                if "method" in cache_df:
                    cached_duration = dict(zip(cache_df["method"].astype(str),
                                               cache_df.get("duration_s", pd.Series(dtype=float))))
                print(f"[macsa sweep] loaded cache: {sweep_routes_path.name}")
            else:
                print("[macsa sweep] cache ignored: iteration count mismatch")
        except Exception as exc:
            print(f"[macsa sweep] cache ignored: {exc}")

    MACSA_SWEEP_ROWS = []
    MACSA_SWEEP_ROUTES = {}

    def macsa_save_sweep_progress():
        df = pd.DataFrame(MACSA_SWEEP_ROWS).sort_values("alpha").round(6)
        save_paper_table(df, MACSA_SWEEP_STEM)
        save_paper_routes(MACSA_SWEEP_STEM, MACSA_SWEEP_ROUTES,
                          MACSA_COORDS, MACSA_STREET_ADJ,
                          meta={"scenario": MACSA_SCENARIO_NAME,
                                "ref_method": MACSA_REF_METHOD,
                                "alpha_grid": MACSA_ALPHA_GRID,
                                "bco_iterations": MACSA_SWEEP_BCO_ITERATIONS,
                                "bco_bees": MACSA_SWEEP_BEES,
                                "seed": MACSA_SWEEP_SEED,
                                "adj_target": MACSA_ADJ_TARGET_FROM_MACSA,
                                "adj_objective": "target"})
        return df

    for alpha in tqdm(MACSA_ALPHA_GRID, desc="MACSA Our NBCO alpha sweep"):
        label = macsa_alpha_label(alpha)
        if label in cached_routes and not MACSA_SWEEP_FORCE_RERUN:
            print(f"  [cache] {label}")
            routes = macsa_pad_routes(cached_routes[label], MACSA_SPEC["n_routes"], MACSA_SPEC["max_route_len"])
            duration = float(cached_duration.get(label, np.nan)) if label in cached_duration else np.nan
            source = "our_nbco_alpha_sweep_cached"
        else:
            print(f"  [run] {label}: alpha={alpha:.1f}, iters={MACSA_SWEEP_BCO_ITERATIONS}", flush=True)
            routes, duration = macsa_run_our_nbco(
                MACSA_SEED_ROUTES, tensors=MACSA_TENSORS, spec=MACSA_SPEC,
                alpha=float(alpha), adj_target=MACSA_ADJ_TARGET_FROM_MACSA,
                adj_objective="target", n_iterations=MACSA_SWEEP_BCO_ITERATIONS,
                n_bees=MACSA_SWEEP_BEES, seed=MACSA_SWEEP_SEED,
                force_cpu=MACSA_FORCE_CPU)
            source = "our_nbco_alpha_sweep"
        row, scored = macsa_score_routes(
            label, source, routes, seed_routes=MACSA_SEED_ROUTES,
            tensors=MACSA_TENSORS, spec=MACSA_SPEC,
            alpha=float(alpha), adj_target=MACSA_ADJ_TARGET_FROM_MACSA,
            adj_objective="target")
        row.update(duration_s=duration, alpha=float(alpha), run_alpha=float(alpha),
                   n_iterations=int(MACSA_SWEEP_BCO_ITERATIONS),
                   macsa_adj_target=MACSA_ADJ_TARGET_FROM_MACSA,
                   beats_macsa_rtt_wmc=bool(
                       (float(row["RTT"]) < float(MACSA_TABLEB_DF.loc[MACSA_TABLEB_DF["method"] == "MACSA", "RTT"].iloc[0])) and
                       (float(row["WMC"]) < float(MACSA_TABLEB_DF.loc[MACSA_TABLEB_DF["method"] == "MACSA", "WMC"].iloc[0]))))
        MACSA_SWEEP_ROWS = macsa_upsert_row(MACSA_SWEEP_ROWS, row)
        MACSA_SWEEP_ROUTES[label] = scored
        MACSA_SWEEP_DF = macsa_save_sweep_progress()
        print(f"    -> RTT={row['RTT']:.0f} WMC={row['WMC']:.2f} "
              f"ATT={row['ATT']:.2f} adj={row['adj_vs_seed']:.3f} cost={row['cost']:.3f} "
              f"duration={duration:.1f}s", flush=True)

    MACSA_SWEEP_DF = pd.DataFrame(MACSA_SWEEP_ROWS).sort_values("alpha").round(6)
    display(MACSA_SWEEP_DF)


In [ ]:
# Pick the best Our NBCO sweep solution that beats MACSA on both RTT and WMC,
# then build the comparison table: all Table-B methods + that single Our solution.
if not SUITE.run.macsa or MACSA_SWEEP_DF.empty:
    MACSA_COMPARISON_DF = pd.DataFrame()
    MACSA_COMPARISON_ROUTES = {}
    print("MACSA comparison skipped: no sweep rows.")
else:
    macsa_ref_row = MACSA_TABLEB_DF[MACSA_TABLEB_DF["method"] == "MACSA"].iloc[0]
    MACSA_BEST_SWEEP_ROW = macsa_select_best_sweep_row(MACSA_SWEEP_DF, macsa_ref_row)
    MACSA_BEST_SWEEP_LABEL = str(MACSA_BEST_SWEEP_ROW["method"])
    MACSA_BEST_ALPHA = float(MACSA_BEST_SWEEP_ROW["alpha"])
    MACSA_BEST_BEATS_MACSA = bool(MACSA_BEST_SWEEP_ROW["beats_macsa_rtt_wmc"])
    MACSA_BEST_COMPARE_LABEL = f"Our NBCO best (alpha={MACSA_BEST_ALPHA:.1f}, iter={MACSA_SWEEP_BCO_ITERATIONS})"

    best_compare_row, best_compare_routes = macsa_score_routes(
        MACSA_BEST_COMPARE_LABEL, "our_nbco_alpha_sweep_best",
        MACSA_SWEEP_ROUTES[MACSA_BEST_SWEEP_LABEL],
        seed_routes=MACSA_SEED_ROUTES, tensors=MACSA_TENSORS, spec=MACSA_SPEC,
        alpha=MACSA_TABLEB_EVAL_ALPHA,
        adj_target=MACSA_TABLEB_EVAL_ADJ_TARGET,
        adj_objective=MACSA_TABLEB_EVAL_ADJ_OBJECTIVE)
    best_compare_row.update(alpha=MACSA_BEST_ALPHA,
                            run_alpha=MACSA_BEST_ALPHA,
                            n_iterations=int(MACSA_SWEEP_BCO_ITERATIONS),
                            macsa_adj_target=MACSA_ADJ_TARGET_FROM_MACSA,
                            beats_macsa_rtt_wmc=MACSA_BEST_BEATS_MACSA,
                            selected_from=MACSA_BEST_SWEEP_LABEL,
                            selection_rule="min RTT/MACSA_RTT + WMC/MACSA_WMC among rows beating MACSA on both")

    MACSA_COMPARISON_ROUTES = dict(MACSA_TABLEB_ROUTES)
    MACSA_COMPARISON_ROUTES[MACSA_BEST_COMPARE_LABEL] = best_compare_routes
    MACSA_COMPARISON_DF = pd.concat(
        [MACSA_TABLEB_DF, pd.DataFrame([best_compare_row])], ignore_index=True, sort=False).round(6)
    save_paper_table(MACSA_COMPARISON_DF, MACSA_COMPARISON_STEM)
    save_paper_routes(MACSA_COMPARISON_STEM, MACSA_COMPARISON_ROUTES,
                      MACSA_COORDS, MACSA_STREET_ADJ,
                      meta={"scenario": MACSA_SCENARIO_NAME,
                            "ref_method": MACSA_REF_METHOD,
                            "article_methods": [MACSA_METHOD_TITLE[k] for k in MACSA_METHOD_ORDER],
                            "best_our_method": MACSA_BEST_COMPARE_LABEL,
                            "best_sweep_method": MACSA_BEST_SWEEP_LABEL,
                            "best_beats_macsa_rtt_wmc": MACSA_BEST_BEATS_MACSA,
                            "bco_iterations": MACSA_SWEEP_BCO_ITERATIONS,
                            "sweep_stem": MACSA_SWEEP_STEM,
                            "cap_mode_included": False})
    print("Best Our NBCO sweep solution:", {
        "method": MACSA_BEST_SWEEP_LABEL,
        "alpha": MACSA_BEST_ALPHA,
        "beats_macsa_rtt_wmc": MACSA_BEST_BEATS_MACSA,
        "RTT": float(MACSA_BEST_SWEEP_ROW["RTT"]),
        "WMC": float(MACSA_BEST_SWEEP_ROW["WMC"]),
        "ATT": float(MACSA_BEST_SWEEP_ROW["ATT"]),
        "adj": float(MACSA_BEST_SWEEP_ROW["adj_vs_seed"]),
    })
    display(MACSA_COMPARISON_DF)


In [ ]:
# Visualize only Our NBCO alpha-sweep solutions.
if not SUITE.run.macsa or MACSA_SWEEP_DF.empty:
    print("MACSA our-only visualization skipped: no sweep rows.")
else:
    sweep_order = [str(row["method"]) for _, row in MACSA_SWEEP_DF.sort_values("alpha").iterrows()]
    sweep_routes_ordered = {name: MACSA_SWEEP_ROUTES[name] for name in sweep_order}
    sweep_rows_by_method = {str(row["method"]): row.to_dict()
                            for _, row in MACSA_SWEEP_DF.iterrows()}

    fig = macsa_draw_grid(
        routes=sweep_routes_ordered, rows_by_method=sweep_rows_by_method,
        coords=MACSA_COORDS, street_adj=MACSA_STREET_ADJ, demand=MACSA_DEMAND,
        diff=False, ref_key=None, ref_routes=MACSA_SEED_ROUTES,
        include_demand=True, include_ref=False, ncol=4,
        title="Mandl-8 MACSA: Our NBCO alpha sweep, plain route sets")
    MACSA_SWEEP_PLAIN_PATH = macsa_save_fig(fig, MACSA_SWEEP_STEM, "viz_plain")

    fig = macsa_draw_grid(
        routes=sweep_routes_ordered, rows_by_method=sweep_rows_by_method,
        coords=MACSA_COORDS, street_adj=MACSA_STREET_ADJ, demand=MACSA_DEMAND,
        diff=True, ref_key=MACSA_REF_METHOD, ref_routes=MACSA_SEED_ROUTES,
        include_demand=True, include_ref=False, ncol=4,
        title="Mandl-8 MACSA: Our NBCO alpha sweep, diff vs original")
    MACSA_SWEEP_DIFF_PATH = macsa_save_fig(fig, MACSA_SWEEP_STEM, "viz_diff")
    macsa_display_image(MACSA_SWEEP_PLAIN_PATH)
    macsa_display_image(MACSA_SWEEP_DIFF_PATH)


In [ ]:
# Visualize all paper Table-B solutions plus the selected best Our NBCO solution.
if not SUITE.run.macsa or MACSA_COMPARISON_DF.empty:
    print("MACSA comparison visualization skipped: no comparison table.")
else:
    comparison_rows_by_method = {str(row["method"]): row.to_dict()
                                 for _, row in MACSA_COMPARISON_DF.iterrows()}
    fig = macsa_draw_grid(
        routes=MACSA_COMPARISON_ROUTES, rows_by_method=comparison_rows_by_method,
        coords=MACSA_COORDS, street_adj=MACSA_STREET_ADJ, demand=MACSA_DEMAND,
        diff=False, ref_key=MACSA_REF_METHOD, ref_routes=MACSA_SEED_ROUTES,
        include_demand=True, include_ref=True, ncol=5,
        title="Mandl-8 MACSA Table B: paper methods + best Our NBCO")
    MACSA_COMPARISON_PLAIN_PATH = macsa_save_fig(fig, MACSA_COMPARISON_STEM, "viz_plain")

    fig = macsa_draw_grid(
        routes=MACSA_COMPARISON_ROUTES, rows_by_method=comparison_rows_by_method,
        coords=MACSA_COORDS, street_adj=MACSA_STREET_ADJ, demand=MACSA_DEMAND,
        diff=True, ref_key=MACSA_REF_METHOD, ref_routes=MACSA_SEED_ROUTES,
        include_demand=True, include_ref=True, ncol=5,
        title="Mandl-8 MACSA Table B: paper methods + best Our NBCO, diff vs original")
    MACSA_COMPARISON_DIFF_PATH = macsa_save_fig(fig, MACSA_COMPARISON_STEM, "viz_diff")
    macsa_display_image(MACSA_COMPARISON_PLAIN_PATH)
    macsa_display_image(MACSA_COMPARISON_DIFF_PATH)
